# Feature selection for length of stay with no treatment

In [1]:
# Turn warnings off to keep notebook tidy
import warnings
warnings.filterwarnings("ignore")

import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from xgboost import XGBRegressor
from sklearn.metrics import r2_score
from sklearn.model_selection import StratifiedKFold

## Load data, filter and create k-fold

In [2]:
features_for_selection = [
    'stroke_team',
    'age',
    'male',
    'discharge_destination',
    'ethnicity',
    'onset_to_arrival_time',
    'arrival_to_scan_time',
    'onset_known',
    'precise_onset_known',
    'onset_during_sleep',
    'arrive_by_ambulance',
    'infarction',
    'congestive_heart_failure',
    'hypertension',
    'atrial_fibrillation',
    'diabetes',
    'prior_stroke_tia',
    'afib_antiplatelet',
    'afib_anticoagulant',
    'any_afib_diagnosis',
    'prior_disability',
    'stroke_severity',
    'random_1',
    'random_2',
    'random_3'
]

In [3]:
all_data = pd.read_csv("../../data/sam3/cleaned_data.csv", low_memory=False)

# Add three 'random' columns to the DataFrame with random values for each row
all_data['random_1'] = np.random.rand(len(all_data))
all_data['random_2'] = np.random.binomial(1, 0.5, len(all_data))
all_data['random_3'] = np.random.randint(1, 11, len(all_data))

# List all_data columns
all_data_columns = all_data.columns.tolist()
print(all_data_columns)

# Print size of all_data
print(f"Size of all_data: {all_data.shape}")

['stroke_team', 'age', 'male', 'ethnicity', 'onset_to_arrival_time', 'onset_known', 'precise_onset_known', 'onset_during_sleep', 'arrive_by_ambulance', 'call_to_ambulance_arrival_time', 'ambulance_on_scene_time', 'ambulance_travel_to_hospital_time', 'ambulance_wait_time_at_hospital', 'month', 'year', 'weekday', 'arrival_time_3_hour_period', 'infarction', 'lvo', 'arrival_to_scan_time', 'thrombolysis', 'thrombolysis_induced_haemorrhage', 'scan_to_thrombolysis_time', 'arrival_to_thrombolysis_time', 'onset_to_thrombolysis_time', 'thrombectomy', 'arrival_to_thrombectomy_referral_time', 'arrival_to_thrombectomy_time', 'onset_to_thrombectomy_time', 'onset_to_thrombolysis_time ', 'ai_aspects', 'aspects_score', 'perfusion_imaging_used', 'transfer_time', 'congestive_heart_failure', 'hypertension', 'atrial_fibrillation', 'diabetes', 'prior_stroke_tia', 'afib_antiplatelet', 'afib_anticoagulant', 'afib_vit_k_anticoagulant', 'afib_doac_anticoagulant', 'afib_heparin_anticoagulant', 'new_afib_diagnosi

Filter data to teams with at least 300 admissions and 10 thrombolysis use

In [4]:
keep = []
groups = all_data.groupby('stroke_team') # creates a new object of groups of data

for index, group_df in groups: # each group has an index and a dataframe of data
    
    # Skip if total admission less than 300 or total thrombolysis < 10
    if (group_df.shape[0] < 300) or (group_df['thrombolysis'].sum() < 10):
        continue
    
    else: 
        keep.append(group_df)

# Concatenate output
data = pd.DataFrame()
data = pd.concat(keep)

n_patients_after_admission_restrictions = data.shape[0]
# Print the number of patients before and after applying the admission restrictions
print(f"Number of patients before admission restrictions: {all_data.shape[0]}")
print(f"Number of patients after admission restrictions: {n_patients_after_admission_restrictions}")
print("Difference in number of patients: ", all_data.shape[0] - n_patients_after_admission_restrictions)

Number of patients before admission restrictions: 452863
Number of patients after admission restrictions: 452115
Difference in number of patients:  748


In [5]:
# Drop any rows with no length of stay data
data = data.dropna(subset=['length_of_stay'])

# Remove rows where thrombolysis or thrombectomy is 1
data = data[(data['thrombolysis'] != 1) & (data['thrombectomy'] != 1)]

k-fold splits

In [6]:
# Create 5-fold data stratified by stroke team and discharge disability
strat = data['stroke_team'].map(str)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
skf.get_n_splits(data, strat.values)

# Put in NumPy arrays
X = data.drop(columns=['length_of_stay']).values
y = data['length_of_stay'].values
X_col_names = list(data.drop(columns=['length_of_stay']).columns)
y_col_name = 'length_of_stay'

# Loop through the k-fold splits

train_data = []
test_data = []

counter = 0
for train_index, test_index in skf.split(X, y):  

    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]
    
    # Create a DataFrame for train and test data
    train_df = pd.DataFrame(X_train, columns=X_col_names)
    train_df[y_col_name] = y_train
    
    test_df = pd.DataFrame(X_test, columns=X_col_names)
    test_df[y_col_name] = y_test
    
    # Append to the list
    train_data.append(train_df)
    test_data.append(test_df)

## Single feature prediction

In [7]:
from pathlib import Path

# Create list to store accuracies and chosen features
r2_by_feature_number = []
ci_lower_by_feature_number = []
ci_upper_by_feature_number = []

num_features = len(features_for_selection)
available_features = features_for_selection.copy()

# Loop through available features
for feature in available_features:

    features_to_use = [feature]  # Use only the current feature for this iteration
  
    # Set up a list to hold R-squared results for this feature for each kfold
    feature_r2_kfold = []
    
    # Loop through k folds
    for k_fold in range(5):

        # Get k fold split
        train = train_data[k_fold]
        test = test_data[k_fold]

        # Get X and y
        X_train = train.drop('length_of_stay', axis=1)
        X_test = test.drop('length_of_stay', axis=1)
        y_train = train['length_of_stay']
        y_test = test['length_of_stay']

        # Restrict features
        X_train = X_train[features_to_use]
        X_test = X_test[features_to_use]

        # One hot encode hospitals if hospital in features used
        if 'stroke_team' in features_to_use:
            X_train_hosp = pd.get_dummies(
                X_train['stroke_team'], prefix = 'team')
            X_train = pd.concat([X_train, X_train_hosp], axis=1)
            X_train.drop('stroke_team', axis=1, inplace=True)
            X_test_hosp = pd.get_dummies(
                X_test['stroke_team'], prefix = 'team')
            X_test = pd.concat([X_test, X_test_hosp], axis=1)
            X_test.drop('stroke_team', axis=1, inplace=True)

        # One hot encode ethnicity if in features used
        if 'ethnicity' in features_to_use:
            X_train_eth = pd.get_dummies(
                X_train['ethnicity'], prefix = 'eth')
            X_train = pd.concat([X_train, X_train_eth], axis=1)
            X_train.drop('ethnicity', axis=1, inplace=True)
            X_test_eth = pd.get_dummies(
                X_test['ethnicity'], prefix = 'eth')
            X_test = pd.concat([X_test, X_test_eth], axis=1)
            X_test.drop('ethnicity', axis=1, inplace=True)

        # One hot encode discharge destination if in features used
        if 'discharge_destination' in features_to_use:
            X_train_dest = pd.get_dummies(
                X_train['discharge_destination'], prefix = 'dest')
            X_train = pd.concat([X_train, X_train_dest], axis=1)
            X_train.drop('discharge_destination', axis=1, inplace=True)
            X_test_dest = pd.get_dummies(
                X_test['discharge_destination'], prefix = 'dest')
            X_test = pd.concat([X_test, X_test_dest], axis=1)
            X_test.drop('discharge_destination', axis=1, inplace=True)

        # Define model
        model = XGBRegressor(
            verbosity=0,
            random_state=42,
            objective='reg:squarederror'
        )

        # Ensure XGBoost-compatible dtypes
        X_train = X_train.apply(pd.to_numeric, errors='coerce')
        X_test = X_test.apply(pd.to_numeric, errors='coerce')
        y_train = pd.to_numeric(y_train, errors='coerce')
        y_test = pd.to_numeric(y_test, errors='coerce')

        # Drop rows with missing target values
        train_mask = y_train.notna()
        test_mask = y_test.notna()
        X_train = X_train.loc[train_mask]
        y_train = y_train.loc[train_mask]
        X_test = X_test.loc[test_mask]
        y_test = y_test.loc[test_mask]

        # Fit model
        model.fit(X_train, y_train)

        # Predict and score with R-squared
        y_pred = model.predict(X_test)
        r2 = r2_score(y_test, y_pred)
        feature_r2_kfold.append(r2)
    
    # Get average result from all k-fold splits
    feature_r2_mean = np.mean(feature_r2_kfold)
    r2_by_feature_number.append(feature_r2_mean)

    # Get 95% confidence interval for the mean R-squared
    ci_lower = np.percentile(feature_r2_kfold, 2.5)
    ci_upper = np.percentile(feature_r2_kfold, 97.5)
    ci_lower_by_feature_number.append(ci_lower)
    ci_upper_by_feature_number.append(ci_upper)

    # Print results for this feature
    print(f"Feature: {feature}, Mean R-squared: {feature_r2_mean:.4f}, 95% CI: ({ci_lower:.4f}, {ci_upper:.4f})")

results = pd.DataFrame({
    'Feature': available_features,
    'Mean R-squared': r2_by_feature_number,
    'lower_95_ci': ci_lower_by_feature_number,
    'upper_95_ci': ci_upper_by_feature_number
})

# Sort results by Mean R-squared in descending order
results = results.sort_values(by='Mean R-squared', ascending=False).reset_index(drop=True)

# Save results to CSV
output_dir = Path("./output")
output_dir.mkdir(parents=True, exist_ok=True)
results.to_csv(output_dir / "single_feature_selection_los_no_treatment_results.csv", index=False)


Feature: stroke_team, Mean R-squared: 0.0234, 95% CI: (0.0226, 0.0242)
Feature: age, Mean R-squared: 0.0060, 95% CI: (0.0056, 0.0064)
Feature: male, Mean R-squared: 0.0004, 95% CI: (0.0003, 0.0007)
Feature: discharge_destination, Mean R-squared: 0.0586, 95% CI: (0.0569, 0.0597)
Feature: ethnicity, Mean R-squared: 0.0013, 95% CI: (0.0010, 0.0016)
Feature: onset_to_arrival_time, Mean R-squared: 0.0003, 95% CI: (0.0003, 0.0005)
Feature: arrival_to_scan_time, Mean R-squared: 0.0075, 95% CI: (0.0067, 0.0084)
Feature: onset_known, Mean R-squared: 0.0001, 95% CI: (0.0000, 0.0002)
Feature: precise_onset_known, Mean R-squared: 0.0016, 95% CI: (0.0015, 0.0017)
Feature: onset_during_sleep, Mean R-squared: 0.0001, 95% CI: (0.0000, 0.0002)
Feature: arrive_by_ambulance, Mean R-squared: 0.0205, 95% CI: (0.0195, 0.0215)
Feature: infarction, Mean R-squared: 0.0110, 95% CI: (0.0100, 0.0120)
Feature: congestive_heart_failure, Mean R-squared: 0.0003, 95% CI: (0.0001, 0.0004)
Feature: hypertension, Mean R-

In [8]:
results

,Feature,Mean R-squared,lower_95_ci,upper_95_ci
0,stroke_severity,0.111209,0.109689,0.112824
1,discharge_destination,0.058586,0.056933,0.059670
2,stroke_team,0.023374,0.022575,0.024250
3,arrive_by_ambulance,0.020482,0.019518,0.021491
4,infarction,0.011040,0.009986,0.011979
5,prior_disability,0.009770,0.009189,0.010620
6,arrival_to_scan_time,0.007480,0.006651,0.008364
7,age,0.006018,0.005592,0.006417
8,precise_onset_known,0.001579,0.001499,0.001703
9,any_afib_diagnosis,0.001565,0.001401,0.001792


## Feature selection

In [9]:
# Create list to store accuracies and chosen features
r2_by_feature_number = []
r2_by_feature_number_kfold = []
chosen_features = []

num_features = len(features_for_selection)
available_features = features_for_selection.copy()

# Loop through number of features
for i in range (num_features):
    
    # Reset best feature and accuracy
    best_result = 0
    best_feature = ''
    
    # Loop through available features
    for feature in available_features:

        # Create copy of already chosen features to avoid original being changed
        features_to_use = chosen_features.copy()
        # Create a list of features from features already chosen + 1 new feature
        features_to_use.append(feature)
        
        # Set up a list to hold R-squared results for this feature for each kfold
        feature_r2_kfold = []
        
        # Loop through k folds
        for k_fold in range(5):

            # Get k fold split
            train = train_data[k_fold]
            test = test_data[k_fold]

            # Get X and y
            X_train = train.drop('length_of_stay', axis=1)
            X_test = test.drop('length_of_stay', axis=1)
            y_train = train['length_of_stay']
            y_test = test['length_of_stay']

            # Restrict features
            X_train = X_train[features_to_use]
            X_test = X_test[features_to_use]
    
            # One hot encode hospitals if hospital in features used
            if 'stroke_team' in features_to_use:
                X_train_hosp = pd.get_dummies(
                    X_train['stroke_team'], prefix = 'team')
                X_train = pd.concat([X_train, X_train_hosp], axis=1)
                X_train.drop('stroke_team', axis=1, inplace=True)
                X_test_hosp = pd.get_dummies(
                    X_test['stroke_team'], prefix = 'team')
                X_test = pd.concat([X_test, X_test_hosp], axis=1)
                X_test.drop('stroke_team', axis=1, inplace=True)

            # One hot encode ethnicity if in features used
            if 'ethnicity' in features_to_use:
                X_train_eth = pd.get_dummies(
                    X_train['ethnicity'], prefix = 'eth')
                X_train = pd.concat([X_train, X_train_eth], axis=1)
                X_train.drop('ethnicity', axis=1, inplace=True)
                X_test_eth = pd.get_dummies(
                    X_test['ethnicity'], prefix = 'eth')
                X_test = pd.concat([X_test, X_test_eth], axis=1)
                X_test.drop('ethnicity', axis=1, inplace=True)

            # One hot encode discharge destination if in features used
            if 'discharge_destination' in features_to_use:
                X_train_dis = pd.get_dummies(
                    X_train['discharge_destination'], prefix = 'dis')
                X_train = pd.concat([X_train, X_train_dis], axis=1)
                X_train.drop('discharge_destination', axis=1, inplace=True)
                X_test_dis = pd.get_dummies(
                    X_test['discharge_destination'], prefix = 'dis')
                X_test = pd.concat([X_test, X_test_dis], axis=1)
                X_test.drop('discharge_destination', axis=1, inplace=True)

            # Define model
            model = XGBRegressor(
                verbosity=0,
                random_state=42,
                objective='reg:squarederror'
            )

            # Fit model
            # Ensure XGBoost-compatible dtypes (the k-fold DataFrames are object-typed)
            X_train = X_train.apply(pd.to_numeric, errors='coerce')
            X_test = X_test.apply(pd.to_numeric, errors='coerce')
            y_train = pd.to_numeric(y_train, errors='coerce')

            model.fit(X_train, y_train)

            # Get predicted y category
            y_pred = model.predict(X_test)

            # Get R-squared for predicted values
            r2 = r2_score(y_test, y_pred)
            feature_r2_kfold.append(r2)
        
        # Get average result from all k-fold splits
        feature_r2_mean = np.mean(feature_r2_kfold)
    
        # Update chosen feature and result if this feature is a new best
        if feature_r2_mean > best_result:
            best_result = feature_r2_mean
            best_result_kfold = feature_r2_kfold
            best_feature = feature
            
    # k-fold splits are complete    
    # Add mean accuracy and AUC to record of accuracy by feature number
    r2_by_feature_number.append(best_result)
    r2_by_feature_number_kfold.append(best_result_kfold)
    chosen_features.append(best_feature)
    available_features.remove(best_feature)
            
    print (f'Feature {i+1:2.0f}: {best_feature}, R-squared: {best_result:0.3f}')

Feature  1: stroke_severity, R-squared: 0.111
Feature  2: discharge_destination, R-squared: 0.192
Feature  3: stroke_team, R-squared: 0.237
Feature  4: prior_disability, R-squared: 0.270
Feature  5: age, R-squared: 0.279
Feature  6: infarction, R-squared: 0.284
Feature  7: onset_to_arrival_time, R-squared: 0.287
Feature  8: arrive_by_ambulance, R-squared: 0.289
Feature  9: arrival_to_scan_time, R-squared: 0.290
Feature 10: male, R-squared: 0.292
Feature 11: afib_anticoagulant, R-squared: 0.292
Feature 12: onset_known, R-squared: 0.292
Feature 13: any_afib_diagnosis, R-squared: 0.292
Feature 14: random_2, R-squared: 0.292
Feature 15: prior_stroke_tia, R-squared: 0.292
Feature 16: diabetes, R-squared: 0.292
Feature 17: onset_during_sleep, R-squared: 0.292
Feature 18: hypertension, R-squared: 0.292
Feature 19: ethnicity, R-squared: 0.292
Feature 20: afib_antiplatelet, R-squared: 0.292
Feature 21: precise_onset_known, R-squared: 0.292
Feature 22: congestive_heart_failure, R-squared: 0.292


In [10]:
# Create a DataFrame to hold the results
results_df = pd.DataFrame({
    'Feature Number': list(range(1, num_features + 1)),
    'Chosen Feature': chosen_features,
    'Mean r2': r2_by_feature_number,
})

# Save to output folder
results_df.to_csv('./output/feature_selection_los_no_treatment_results.csv', index=False)

In [11]:
results_df

,Feature Number,Chosen Feature,Mean r2
0,1,stroke_severity,0.111209
1,2,discharge_destination,0.192151
2,3,stroke_team,0.237019
3,4,prior_disability,0.269896
4,5,age,0.278685
5,6,infarction,0.284043
6,7,onset_to_arrival_time,0.287081
7,8,arrive_by_ambulance,0.289428
8,9,arrival_to_scan_time,0.290343
9,10,male,0.292260
